# Fingerprint Similarity Heatmaps

This notebook builds similarity matrices between benign devices using JA4+ fingerprint values.

For each fingerprint combination, the notebook:

1. Builds one set of observed fingerprint bundles per device.
2. Keeps every device that has at least one valid bundle for that combination.
3. Compares every pair of devices with Jaccard similarity.
4. Saves a heatmap with all device names explicitly shown on both axes.

The resulting values range from `0` to `1`: `0` means that two devices do not share any observed bundles, while `1` means that their sets of bundles are identical.


In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')


## Configuration

This section locates the repository root, defines the input CSV file, and creates the output directory where the heatmaps will be saved.

In [2]:
# Resolve the project root dynamically so the notebook works from any subfolder.
cwd = Path.cwd().resolve()
ROOT = next(path for path in [cwd, *cwd.parents] if (path / '1_Data').exists())

# Input data and output folder for the generated figures.
fingerprints_path = ROOT / '1_Data' / 'raw_fingerprints' / 'raw_features.csv'
OUTPUT_DIR = ROOT / '3_Results' / 'similarity_matrices'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input directory: {fingerprints_path}')
print(f'Output directory: {OUTPUT_DIR}')


Input directory: D:\TFG_GemmaBeatrizVate\1_Data\raw_fingerprints\raw_features.csv
Output directory: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices


## Load Dataset

The source file contains the extracted fingerprint fields and the device label used to group observations.

In [3]:
# low_memory=False avoids mixed-type inference warnings on wide CSV files.
df_benign = pd.read_csv(fingerprints_path, low_memory=False)
print(f'Rows: {len(df_benign)}')
print(f'Columns: {len(df_benign.columns)}')


Rows: 162362
Columns: 54


## Fingerprint Combinations and Helper Functions

Each key is the label used in the plots and each value is the list of columns that will be combined into one fingerprint bundle per row. The helper functions below clean values, build one set of bundles per device, compute the Jaccard similarity matrix, and save the heatmap.

Each matrix keeps every device that has at least one valid fingerprint bundle for that specific combination. The heatmap explicitly draws every device name on both axes, even when there are many labels.

In [4]:
# Fingerprint combinations evaluated in the similarity analysis.
FULL_FINGERPRINT_GROUPS = {
    'JA4': ['ja4'],
    'JA4+SNI': ['ja4', 'sni'],
    'JA4S': ['ja4s'],
    'JA4X': ['ja4x'],
    'JA4+JA4S': ['ja4', 'ja4s'],
    'JA4+JA4X': ['ja4', 'ja4x'],
    'JA4T+JA4TS': ['ja4t', 'ja4ts'],
    'JA4+JA4S+JA4TS': ['ja4', 'ja4s', 'ja4ts'],
    'JA4+JA4S+JA4T+JA4TS': ['ja4', 'ja4s', 'ja4t', 'ja4ts'],
    'JA4+': ['ja4', 'ja4s', 'ja4x', 'ja4t', 'ja4ts'],
}

def normalize_label(series):
    """Standardize device names before grouping rows by device."""
    return series.fillna('unknown').astype(str).str.strip().str.lower()

def is_valid_value(value):
    """Return False for missing or placeholder fingerprint values."""
    if pd.isna(value):
        return False
    text = str(value).strip().lower()
    return text not in {'', '(empty)', 'unknown', 'nan'}

def safe_name(value):
    """Create filesystem-safe filenames from plot labels."""
    return ''.join(ch if ch.isalnum() or ch in '-_+' else '_' for ch in value)

def device_signature_sets(df, fingerprint_cols, label_col='device_name'):
    """Build a set of unique fingerprint bundles for each valid device.

    A bundle is built row by row by joining the selected fingerprint columns as
    `column=value` pairs. Empty, missing, or placeholder values are ignored so
    that only meaningful fingerprint evidence contributes to the comparison.
    Devices without any valid bundle for the selected combination are excluded
    from that specific matrix.
    """
    usable_cols = [col for col in fingerprint_cols if col in df.columns]
    working = df.copy()
    working[label_col] = normalize_label(working[label_col])

    def row_to_bundle(row):
        # A bundle represents the complete fingerprint combination observed in one row.
        parts = []
        for col in usable_cols:
            value = row.get(col)
            if is_valid_value(value):
                parts.append(f'{col}={str(value).strip().lower()}')
        return ' | '.join(parts) if parts else None

    # Rows without any valid value for the selected columns cannot inform similarity.
    working['signature_bundle'] = working.apply(row_to_bundle, axis=1)
    working = working[working['signature_bundle'].notna()].copy()

    output = {}
    for device_name, group in working.groupby(label_col, sort=True):
        # Sets remove repeated observations, so similarity measures shared diversity.
        output[device_name] = set(group['signature_bundle'].astype(str))
    return output

def jaccard_similarity(set_a, set_b):
    """Compute |A intersection B| / |A union B| for two signature sets."""
    if not set_a and not set_b:
        return 0.0
    union = set_a | set_b
    if not union:
        return 0.0
    return len(set_a & set_b) / len(union)

def build_similarity_matrix(device_sets):
    """Compare every device against every other device with Jaccard similarity."""
    labels = sorted(device_sets.keys())
    matrix = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for row_label in labels:
        for col_label in labels:
            matrix.loc[row_label, col_label] = jaccard_similarity(device_sets[row_label], device_sets[col_label])
    return matrix

def plot_title(combo_name):
    """Return the required title format for each fingerprint combination."""
    return f'Similarity matrix {combo_name} Fingerprints'

def save_heatmap(matrix_df, output_path, combo_name):
    """Save a balanced heatmap with every device label visible."""
    if matrix_df.empty:
        print(f'Skipping empty heatmap: {combo_name}')
        return

    n_devices = len(matrix_df.index)
    figure_size = max(10, min(13, n_devices * 0.24))

    figure, axis = plt.subplots(figsize=(figure_size, figure_size))
    sns.heatmap(
        matrix_df,
        cmap='Blues',
        annot=False,
        vmin=0,
        vmax=1,
        square=True,
        xticklabels=matrix_df.columns,
        yticklabels=matrix_df.index,
        cbar_kws={'shrink': 0.72, 'pad': 0.02},
        ax=axis,
    )
    axis.set_xticks([index + 0.5 for index in range(n_devices)])
    axis.set_yticks([index + 0.5 for index in range(n_devices)])
    axis.set_xticklabels(matrix_df.columns, rotation=90, ha='center', fontsize=8)
    axis.set_yticklabels(matrix_df.index, rotation=0, fontsize=8)
    axis.set_title(plot_title(combo_name), pad=14, fontsize=13)
    axis.set_xlabel('Device', fontsize=11)
    axis.set_ylabel('Device', fontsize=11)
    axis.tick_params(axis='both', length=0, pad=2)
    figure.subplots_adjust(left=0.28, bottom=0.30, right=0.94, top=0.92)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved: {output_path}')


## Similarity Matrix Summary

Before plotting, this cell builds every matrix and reports a compact summary. The `devices` column counts the devices that have at least one valid bundle for that fingerprint combination. The off-diagonal statistics exclude each device compared with itself.

In [5]:
analysis_results = {}
summary_rows = []

for combo_name, fingerprint_cols in FULL_FINGERPRINT_GROUPS.items():
    # Build the per-device sets and then convert them into a square similarity matrix.
    device_sets = device_signature_sets(df_benign, fingerprint_cols)
    similarity_matrix = build_similarity_matrix(device_sets)
    analysis_results[combo_name] = {
        'sets': device_sets,
        'matrix': similarity_matrix,
    }

    signature_counts = pd.Series({device: len(signatures) for device, signatures in device_sets.items()})
    all_bundles = set().union(*device_sets.values()) if device_sets else set()

    # Off-diagonal values describe similarity between different devices only.
    off_diagonal_values = []
    labels = list(similarity_matrix.index)
    for row_number, row_label in enumerate(labels):
        for col_number, col_label in enumerate(labels):
            if row_number != col_number:
                off_diagonal_values.append(similarity_matrix.loc[row_label, col_label])
    off_diagonal = pd.Series(off_diagonal_values, dtype=float)

    summary_rows.append({
        'fingerprint_combination': combo_name,
        'devices': len(device_sets),
        'unique_signature_bundles': len(all_bundles),
        'avg_bundles_per_device': signature_counts.mean() if not signature_counts.empty else 0,
        'mean_pair_similarity': off_diagonal.mean() if not off_diagonal.empty else 0,
        'max_pair_similarity': off_diagonal.max() if not off_diagonal.empty else 0,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,fingerprint_combination,devices,unique_signature_bundles,avg_bundles_per_device,mean_pair_similarity,max_pair_similarity
0,JA4,51,92,2.470588,0.040105,1.0
1,JA4+SNI,51,292,6.411765,0.038603,1.0
2,JA4S,41,52,2.219512,0.137201,1.0
3,JA4X,18,13,1.444444,0.087364,1.0
4,JA4+JA4S,52,163,4.211538,0.032663,1.0
5,JA4+JA4X,51,111,2.941176,0.039845,1.0
6,JA4T+JA4TS,51,100,3.274510,0.044987,1.0
7,JA4+JA4S+JA4TS,52,290,7.307692,0.025810,1.0
8,JA4+JA4S+JA4T+JA4TS,52,331,7.923077,0.025086,1.0
9,JA4+,52,345,8.250000,0.024846,1.0


## Generate Similarity Matrices

Each heatmap is saved with the title `Similarity matrix <combination> Fingerprints`. The x-axis and y-axis both represent device labels, and the color scale represents the Jaccard similarity score.

In [6]:
for combo_name, result in analysis_results.items():
    benign_matrix = result['matrix']
    combo_slug = safe_name(combo_name.lower())
    output_path = OUTPUT_DIR / f'{combo_slug}_similarity_matrix.png'
    save_heatmap(benign_matrix, output_path, combo_name)


Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4+sni_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4s_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4x_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4+ja4s_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4+ja4x_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4t+ja4ts_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4+ja4s+ja4ts_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4+ja4s+ja4t+ja4ts_similarity_matrix.png
Saved: D:\TFG_GemmaBeatrizVate\3_Results\similarity_matrices\ja4+_similarity_matrix.png
